In [ ]:
!pip install torch-geometric -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 657.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 6.3 MB/s eta 0:00:00


In [ ]:
import torch
from torch_geometric.nn import GCNConv
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("PyG installed successfully!")

PyTorch version: 2.11.0+cpu
CUDA available: False
PyG installed successfully!


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch_geometric.data import Data
import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.preprocessing import StandardScaler

print("All imports successful!")
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

All imports successful!
Device: cpu


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, SAGEConv
from torch_geometric.data import Data
from torch_geometric.utils import train_test_split_edges
import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

print("Imports done!")

Imports done!


In [ ]:
import os

os.makedirs('/content/data', exist_ok=True)

files = [
    'bio-decagon-combo.tar.gz',
    'bio-decagon-mono.tar.gz',
    'bio-decagon-targets.tar.gz',
    'bio-decagon-ppi.tar.gz'
]

base_url = 'http://snap.stanford.edu/decagon'

for f in files:
    out = f'/content/data/{f}'
    if not os.path.exists(out.replace('.tar.gz', '.csv')):
        print(f"Downloading {f}...")
        os.system(f'wget -q {base_url}/{f} -O {out}')
        os.system(f'tar -xzf {out} -C /content/data/')
        print(f"Done: {f}")
    else:
        print(f"Already exists: {f}")

print("\nFiles in data folder:")
print([f for f in os.listdir('/content/data') if f.endswith('.csv')])

Done: bio-decagon-combo.tar.gz
Done: bio-decagon-mono.tar.gz
Done: bio-decagon-targets.tar.gz
Done: bio-decagon-ppi.tar.gz

Files in data folder:
['bio-decagon-combo.csv', '._bio-decagon-ppi.csv', 'bio-decagon-targets.csv', 'bio-decagon-ppi.csv', '._bio-decagon-targets.csv', 'bio-decagon-mono.csv', '._bio-decagon-combo.csv', '._bio-decagon-mono.csv']


In [ ]:
print("Loading real dataset...")
combo   = pd.read_csv('/content/data/bio-decagon-combo.csv')
targets = pd.read_csv('/content/data/bio-decagon-targets.csv')

drug1_col   = 'STITCH 1'
drug2_col   = 'STITCH 2'
se_col      = 'Polypharmacy Side Effect'
se_name_col = 'Side Effect Name'

se_counts      = combo[se_col].value_counts()
valid_se       = se_counts[se_counts >= 500].index
combo_filtered = combo[combo[se_col].isin(valid_se)].copy()

top25_common = se_counts[se_counts >= 500].head(25).index
top25_rare   = se_counts[se_counts >= 500].tail(25).index
mixed_se     = top25_common.tolist() + top25_rare.tolist()

combo_filtered = combo_filtered[
    combo_filtered[se_col].isin(mixed_se)
].copy()

common_se = top25_common.tolist()
rare_se   = top25_rare.tolist()

all_drugs = pd.unique(combo_filtered[[drug1_col, drug2_col]].values.ravel())
drug2idx  = {d: i for i, d in enumerate(all_drugs)}
n_drugs   = len(all_drugs)

print(f"Total side effects available: {len(valid_se)}")
print(f"Using mixed dataset: 25 common + 25 rare = 50 total")
print(f"Total triples: {len(combo_filtered):,}")
print(f"Drugs in subset: {n_drugs}")

print(f"\nTop 5 COMMON side effects:")
for se in top25_common[:5]:
    name  = combo[combo[se_col]==se][se_name_col].iloc[0]
    count = se_counts[se]
    print(f"  {name:<40} {count:,} pairs")

print(f"\nBottom 5 RARE side effects:")
for se in top25_rare[-5:]:
    name  = combo[combo[se_col]==se][se_name_col].iloc[0]
    count = se_counts[se]
    print(f"  {name:<40} {count:,} pairs")


Loading real dataset...
Total side effects available: 963
Using mixed dataset: 25 common + 25 rare = 50 total
Total triples: 566,602
Drugs in subset: 645

Top 5 COMMON side effects:
  arterial pressure NOS decreased          28,568 pairs
  anaemia                                  27,006 pairs
  Difficulty breathing                     26,037 pairs
  nausea                                   25,190 pairs
  neumonia                                 24,430 pairs

Bottom 5 RARE side effects:
  coccydynia                               508 pairs
  neonatal respiratory distress syndrome   507 pairs
  impetigo                                 507 pairs
  Collagen disease                         504 pairs
  sarcoma                                  502 pairs


In [ ]:
# Build drug-protein matrix
prot_targets = targets.copy()
all_prots    = prot_targets.iloc[:, 1].unique()
prot2idx     = {p: i for i, p in enumerate(all_prots)}
n_prots      = len(all_prots)

rows, cols = [], []
for _, row in prot_targets.iterrows():
    drug = row.iloc[0]
    prot = row.iloc[1]
    if drug in drug2idx and prot in prot2idx:
        rows.append(drug2idx[drug])
        cols.append(prot2idx[prot])

drug_prot_adj   = sp.csr_matrix((np.ones(len(rows)), (rows, cols)),
                                 shape=(n_drugs, n_prots))
drug_prot_dense = drug_prot_adj.toarray()

# PCA to 50 dimensions — same as MLP notebook
pca           = PCA(n_components=50, random_state=42)
drug_features = pca.fit_transform(drug_prot_dense)
print(f"Drug features shape: {drug_features.shape}")
print(f"Variance explained:  {pca.explained_variance_ratio_.sum():.1%}")


Drug features shape: (645, 50)
Variance explained:  98.0%


In [ ]:
se_pair_counts = combo_filtered.groupby(se_col).size().sort_values(ascending=False)
all_50_se      = se_pair_counts.index.tolist()

# Top 25 = common, bottom 25 = rare (same split as Decagon notebook)
common_se = all_50_se[:25]
rare_se   = all_50_se[25:]

print(f"Common side effects (top 25): {len(common_se)}")
print(f"  e.g. {se_pair_counts[common_se].head(3).to_dict()}")
print(f"\nRare side effects (bottom 25): {len(rare_se)}")
print(f"  e.g. {se_pair_counts[rare_se].tail(3).to_dict()}")


Common side effects (top 25): 25
  e.g. {'C0020649': 28568, 'C0002871': 27006, 'C0013404': 26037}

Rare side effects (bottom 25): 25
  e.g. {'C0021099': 507, 'C0009326': 504, 'C1261473': 502}


In [ ]:
class LightGNN(nn.Module):
    def __init__(self, in_feats, hidden=256, out_feats=128, dropout=0.3):
        super().__init__()
        self.conv1   = GCNConv(in_feats, hidden)
        self.conv2   = GCNConv(hidden, out_feats)
        self.dropout = dropout

    def encode(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return x

    def decode(self, z, edge_index):
        """
        Score each edge using dot product of node embeddings.
        High score = model thinks these drugs interact.
        """
        src = z[edge_index[0]]
        dst = z[edge_index[1]]
        return (src * dst).sum(dim=1)

    def forward(self, x, edge_index, pred_edges):
        z = self.encode(x, edge_index)
        return self.decode(z, pred_edges)


In [ ]:
from collections import Counter

all_d1 = combo_filtered[drug1_col].map(drug2idx).dropna().astype(int)
all_d2 = combo_filtered[drug2_col].map(drug2idx).dropna().astype(int)
degree_dict = Counter(all_d1.tolist() + all_d2.tolist())

print(f"Degree dictionary built for {len(degree_dict)} drugs")
print(f"Avg degree: {np.mean(list(degree_dict.values())):.1f}")

Degree dictionary built for 645 drugs
Avg degree: 1756.9


In [ ]:
def sample_hard_negatives_v2(d1_ids, d2_ids, n_drugs, degree_dict, pos_set, n_needed, seed=42, window=0.3):
    """
    For each positive pair, sample a negative drug with degree close to
    ONE of the actual drugs in that pair (not the whole group's min/max range).
    """
    np.random.seed(seed)

    all_degrees = np.array([degree_dict.get(d, 0) for d in range(n_drugs)])
    sorted_drug_ids = np.argsort(all_degrees)
    sorted_degrees  = all_degrees[sorted_drug_ids]

    def find_similar_degree_drug(target_degree, exclude_id):
        lo = target_degree * (1 - window)
        hi = target_degree * (1 + window)
        candidates = sorted_drug_ids[(sorted_degrees >= lo) & (sorted_degrees <= hi)]
        candidates = candidates[candidates != exclude_id]
        if len(candidates) == 0:
            return exclude_id
        return candidates[np.random.randint(0, len(candidates))]

    neg_r, neg_c = [], []
    attempts = 0
    n_pos = len(d1_ids)

    while len(neg_r) < n_needed and attempts < n_needed * 20:
        i = np.random.randint(0, n_pos)
        anchor_drug = d1_ids[i]
        anchor_deg  = degree_dict.get(d2_ids[i], 1)

        candidate = find_similar_degree_drug(anchor_deg, exclude_id=anchor_drug)

        r, c = anchor_drug, candidate
        if r != c and (r, c) not in pos_set and (c, r) not in pos_set:
            neg_r.append(r)
            neg_c.append(c)
        attempts += 1

    return np.array(neg_r), np.array(neg_c)

print("sample_hard_negatives_v2 defined!")

sample_hard_negatives_v2 defined!


In [ ]:
# ── Cell: Build base global graph (drug-drug edges EXCLUDING the SE being tested) ──
def build_graph_excluding_pairs(combo_filtered, drug2idx, n_drugs, exclude_pairs_set):
    """
    Builds the message-passing graph from ALL 50 side effects,
    but excludes any edge whose (drug_a, drug_b) pair is in
    exclude_pairs_set — regardless of which side effect it's
    listed under. This is true leak-free exclusion.
    """
    rows, cols = [], []
    for _, row in combo_filtered.iterrows():
        d1 = row[drug1_col]
        d2 = row[drug2_col]
        if d1 in drug2idx and d2 in drug2idx:
            r, c = drug2idx[d1], drug2idx[d2]
            if (r, c) in exclude_pairs_set or (c, r) in exclude_pairs_set:
                continue   # ← skip this specific drug pair entirely
            rows.extend([r, c])
            cols.extend([c, r])

    edge_index = torch.tensor([rows, cols], dtype=torch.long)
    return edge_index

print("build_graph_excluding_pairs defined!")

build_graph_excluding_pairs defined!


In [ ]:
def train_gnn_for_se(se_code, se_name, combo_df, drug2idx,
                     drug_features, n_drugs, degree_dict,
                     n_epochs=10, patience=3, device='cpu',
                     neg_window=0.3):
    se_df  = combo_df[combo_df[se_col] == se_code]
    d1_ids = se_df[drug1_col].map(drug2idx).dropna().astype(int).values
    d2_ids = se_df[drug2_col].map(drug2idx).dropna().astype(int).values

    valid  = (d1_ids < n_drugs) & (d2_ids < n_drugs)
    d1_ids, d2_ids = d1_ids[valid], d2_ids[valid]

    if len(d1_ids) < 50:
        print(f"  Skipping {se_name} — too few pairs ({len(d1_ids)})")
        return None

    n_pos  = len(d1_ids)
    idx    = np.arange(n_pos)
    np.random.shuffle(idx)
    n_val  = max(10, int(n_pos * 0.1))
    n_test = max(10, int(n_pos * 0.1))

    train_idx = idx[n_test + n_val:]
    val_idx   = idx[n_test:n_test + n_val]
    test_idx  = idx[:n_test]

    val_test_idx = np.concatenate([val_idx, test_idx])
    exclude_pairs_set = set(zip(
        d1_ids[val_test_idx].tolist(),
        d2_ids[val_test_idx].tolist()
    ))

    graph_ei = build_graph_excluding_pairs(
        combo_df, drug2idx, n_drugs, exclude_pairs_set
    ).to(device)

    x = torch.tensor(drug_features, dtype=torch.float).to(device)

    pos_set = set(zip(d1_ids.tolist(), d2_ids.tolist()))

    model     = LightGNN(in_feats=drug_features.shape[1], hidden=256, out_feats=128).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=5e-4)
    criterion = nn.BCEWithLogitsLoss()

    best_val_auroc = 0.0
    best_state     = None
    no_improve     = 0

    train_d1, train_d2 = d1_ids[train_idx], d2_ids[train_idx]

    for epoch in range(1, n_epochs + 1):
        model.train()

        neg_r, neg_c = sample_hard_negatives_v2(
            train_d1, train_d2, n_drugs, degree_dict, pos_set,
            n_needed=len(train_idx), seed=epoch, window=neg_window
        )
        neg_r = neg_r[:len(train_idx)]
        neg_c = neg_c[:len(train_idx)]

        all_r  = np.concatenate([train_d1[:len(neg_r)], neg_r])
        all_c  = np.concatenate([train_d2[:len(neg_c)], neg_c])
        labels = np.concatenate([np.ones(len(neg_r)), np.zeros(len(neg_r))])

        pred_ei = torch.tensor(np.vstack([all_r, all_c]), dtype=torch.long).to(device)
        y       = torch.tensor(labels, dtype=torch.float).to(device)

        optimizer.zero_grad()
        logits = model(x, graph_ei, pred_ei)
        loss   = criterion(logits, y)
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            val_d1, val_d2 = d1_ids[val_idx], d2_ids[val_idx]
            neg_vr, neg_vc = sample_hard_negatives_v2(
                val_d1, val_d2, n_drugs, degree_dict, pos_set,
                n_needed=len(val_idx), seed=100 + epoch, window=neg_window
            )
            neg_vr, neg_vc = neg_vr[:len(val_idx)], neg_vc[:len(val_idx)]

            v_r   = np.concatenate([val_d1[:len(neg_vr)], neg_vr])
            v_c   = np.concatenate([val_d2[:len(neg_vc)], neg_vc])
            v_lab = np.concatenate([np.ones(len(neg_vr)), np.zeros(len(neg_vr))])

            v_ei   = torch.tensor(np.vstack([v_r, v_c]), dtype=torch.long).to(device)
            v_log  = model(x, graph_ei, v_ei)
            v_prob = torch.sigmoid(v_log).cpu().numpy()

            val_auroc = roc_auc_score(v_lab, v_prob)

        if val_auroc > best_val_auroc + 0.001:
            best_val_auroc = val_auroc
            best_state     = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve     = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                break

    if best_state:
        model.load_state_dict(best_state)

    model.eval()
    with torch.no_grad():
        te_d1, te_d2 = d1_ids[test_idx], d2_ids[test_idx]
        neg_tr, neg_tc = sample_hard_negatives_v2(
            te_d1, te_d2, n_drugs, degree_dict, pos_set,
            n_needed=len(test_idx), seed=999, window=neg_window
        )
        neg_tr, neg_tc = neg_tr[:len(test_idx)], neg_tc[:len(test_idx)]

        t_r   = np.concatenate([te_d1[:len(neg_tr)], neg_tr])
        t_c   = np.concatenate([te_d2[:len(neg_tc)], neg_tc])
        t_lab = np.concatenate([np.ones(len(neg_tr)), np.zeros(len(neg_tr))])

        t_ei   = torch.tensor(np.vstack([t_r, t_c]), dtype=torch.long).to(device)
        t_log  = model(x, graph_ei, t_ei)
        t_prob = torch.sigmoid(t_log).cpu().numpy()

        test_auroc = roc_auc_score(t_lab, t_prob)
        test_auprc = average_precision_score(t_lab, t_prob)

    return {
        'se_name':  se_name,
        'se_code':  se_code,
        'n_pairs':  n_pos,
        'auroc':    test_auroc,
        'auprc':    test_auprc,
        'best_val': best_val_auroc
    }

print("train_gnn_for_se (final, with neg_window) defined!")

train_gnn_for_se (final, with neg_window) defined!


In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

import time
start_time = time.time()

gnn_results_common = []
gnn_results_rare   = []

print("\n── Common side effects (25) ──")
for i, se_code in enumerate(common_se):
    se_name = combo_filtered[combo_filtered[se_col] == se_code][se_name_col].iloc[0]
    t0 = time.time()
    result  = train_gnn_for_se(se_code, se_name, combo_filtered, drug2idx,
                                drug_features, n_drugs, degree_dict,
                                n_epochs=20, patience=5, neg_window=0.3,
                                device=device)
    elapsed = time.time() - t0
    if result:
        gnn_results_common.append(result)
        print(f"  [{i+1}/25] {se_name:<40} AUROC: {result['auroc']:.3f}  AUPRC: {result['auprc']:.3f}  ({elapsed:.0f}s)")

print("\n── Rare side effects (25) ──")
for i, se_code in enumerate(rare_se):
    se_name = combo_filtered[combo_filtered[se_col] == se_code][se_name_col].iloc[0]
    t0 = time.time()
    result  = train_gnn_for_se(se_code, se_name, combo_filtered, drug2idx,
                                drug_features, n_drugs, degree_dict,
                                n_epochs=20, patience=5, neg_window=0.3,
                                device=device)
    elapsed = time.time() - t0
    if result:
        gnn_results_rare.append(result)
        print(f"  [{i+1}/25] {se_name:<40} AUROC: {result['auroc']:.3f}  AUPRC: {result['auprc']:.3f}  ({elapsed:.0f}s)")

total_time = time.time() - start_time
print(f"\nTotal training time: {total_time/60:.1f} minutes")

Using device: cpu

── Common side effects (25) ──
  [1/25] arterial pressure NOS decreased          AUROC: 0.628  AUPRC: 0.635  (59s)
  [2/25] anaemia                                  AUROC: 0.575  AUPRC: 0.569  (50s)
  [3/25] Difficulty breathing                     AUROC: 0.609  AUPRC: 0.600  (50s)
  [4/25] nausea                                   AUROC: 0.592  AUPRC: 0.598  (64s)
  [5/25] neumonia                                 AUROC: 0.606  AUPRC: 0.602  (55s)
  [6/25] Fatigue                                  AUROC: 0.611  AUPRC: 0.614  (77s)
  [7/25] Pain                                     AUROC: 0.635  AUPRC: 0.638  (73s)
  [8/25] diarrhea                                 AUROC: 0.588  AUPRC: 0.592  (66s)
  [9/25] asthenia                                 AUROC: 0.609  AUPRC: 0.608  (50s)
  [10/25] emesis                                   AUROC: 0.585  AUPRC: 0.587  (78s)
  [11/25] edema extremities                        AUROC: 0.616  AUPRC: 0.615  (66s)
  [12/25] body temperatu

In [ ]:
import numpy as np

common_aurocs = [r['auroc'] for r in gnn_results_common]
common_auprcs = [r['auprc'] for r in gnn_results_common]
rare_aurocs   = [r['auroc'] for r in gnn_results_rare]
rare_auprcs   = [r['auprc'] for r in gnn_results_rare]

print("=" * 50)
print("GNN SUMMARY")
print("=" * 50)
print(f"{'Category':<20} {'AUROC':>8} {'AUPRC':>8}")
print("-" * 50)
print(f"{'Common SEs (25)':<20} {np.mean(common_aurocs):>8.4f} {np.mean(common_auprcs):>8.4f}")
print(f"{'Rare SEs (25)':<20} {np.mean(rare_aurocs):>8.4f} {np.mean(rare_auprcs):>8.4f}")
print(f"{'Overall (50)':<20} {np.mean(common_aurocs+rare_aurocs):>8.4f} {np.mean(common_auprcs+rare_auprcs):>8.4f}")

GNN SUMMARY
Category                AUROC    AUPRC
--------------------------------------------------
Common SEs (25)        0.6012   0.6015
Rare SEs (25)          0.5536   0.5713
Overall (50)           0.5774   0.5864
